# 00 | Smoke Test:環境與資料庫連線驗證
目的:驗證 Python 環境可透過 sqlalchemy 讀取 PostgreSQL raw 層,並取得基準流失率。

In [1]:
import sys
import pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # 把專案根目錄加進模組搜尋路徑,才 import 得到 src/

from src.db import get_engine
import pandas as pd

# GROUP BY attrition_flag : 按流失標記分組。這欄只有兩種值('Existing Customer' / 'Attrited Customer')，所以會分出兩組。
engine = get_engine()
df = pd.read_sql(
    "SELECT attrition_flag, COUNT(*) AS n FROM raw.bank_churners GROUP BY attrition_flag",
    engine,
)
df

,attrition_flag,n
0,Attrited Customer,1627
1,Existing Customer,8500


In [2]:
churn_rate = df.loc[df["attrition_flag"] == "Attrited Customer", "n"].sum() / df["n"].sum()
print(f"基準流失率:{churn_rate:.2%}")

基準流失率:16.07%


In [3]:
# 用 pandas 走一次載入流程，寫到暫存表，與 \copy 版比對
raw_df = pd.read_csv("../data/raw/BankChurners.csv")

# 把末兩欄原始欄名改成短名，因為末兩欄原始欄名超過 PostgreSQL 63 字元上限，截斷後會重複
raw_df.columns.values[-2] = "nb_classifier_prob_1"
raw_df.columns.values[-1] = "nb_classifier_prob_2"

raw_df.to_sql("bank_churners_pandas", engine, schema="raw", if_exists="replace", index=False)
pd.read_sql("SELECT COUNT(*) FROM raw.bank_churners_pandas", engine)
# 預期 10127

,count
0,10127


In [4]:
# 比對型別
pd.read_sql("""
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'raw' AND LOWER(column_name) = 'customer_age'
""", engine)

,table_name,column_name,data_type
0,bank_churners_pandas,Customer_Age,bigint
1,bank_churners,customer_age,text


In [5]:
# 比對完即清掉暫存表，raw 層保持乾淨
from sqlalchemy import text
with engine.begin() as conn:
    conn.execute(text("DROP TABLE raw.bank_churners_pandas"))